# Reasoning Capabilities with Mistral AI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mistralai/cookbook/blob/main/mistral/reasoning/reasoning_capabilities.ipynb)

This notebook explores the reasoning capabilities of Mistral's **Magistral** models, demonstrating chain-of-thought prompting, mathematical reasoning, logical deduction, and code analysis.

Mistral's reasoning models (`magistral-small-latest`, `magistral-medium-latest`) use **Test Time Computation** to explore problems more deeply by generating additional tokens during inference, producing structured thinking traces alongside final answers.

**What you'll learn:**
- How to use Mistral's Magistral reasoning models
- How to parse thinking traces vs. final answers
- Chain-of-thought prompting for complex problems
- Mathematical reasoning, logical deduction, and code analysis
- Comparing results with and without reasoning

In [ ]:
!pip install mistralai==1.12.4

In [ ]:
import os
from mistralai import Mistral

api_key = os.environ.get("MISTRAL_API_KEY")
client = Mistral(api_key=api_key)

REASONING_MODEL = "magistral-medium-latest"
STANDARD_MODEL = "mistral-large-latest"


def chat(prompt, model=REASONING_MODEL, system=None):
    """Send a prompt and return the full response content."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.complete(model=model, messages=messages)
    return response.choices[0].message.content


def chat_with_traces(prompt, model=REASONING_MODEL, system=None):
    """Send a prompt and return thinking traces and final answer separately."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.complete(model=model, messages=messages)
    content = response.choices[0].message.content

    thinking = []
    answer = []
    if isinstance(content, list):
        for block in content:
            if hasattr(block, "type"):
                if block.type == "thinking":
                    thinking.append(block.text)
                elif block.type == "text":
                    answer.append(block.text)
    else:
        answer.append(str(content))

    return {
        "thinking": "\n".join(thinking),
        "answer": "\n".join(answer),
    }

## 1. Understanding Reasoning Model Output

Magistral models split their output into two parts:
- **Thinking blocks** (`type: "thinking"`): The model's internal reasoning traces
- **Text blocks** (`type: "text"`): The final answer

Let's start with a simple example to see how thinking traces work.

In [ ]:
result = chat_with_traces(
    "If a shirt costs $25 after a 20% discount, what was the original price?"
)

print("=== Thinking Traces ===")
print(result["thinking"])
print("\n=== Final Answer ===")
print(result["answer"])

## 2. Chain-of-Thought Prompting

Chain-of-thought (CoT) prompting encourages the model to break down a problem into intermediate steps. While Magistral models naturally produce reasoning traces, explicit CoT instructions can further improve results on complex problems.

In [ ]:
cot_prompt = """A farmer has 3 fields. The first field produces 240 kg of wheat per hectare.
The second field produces 15% more than the first. The third field produces 10% less
than the second. If each field is 5 hectares, what is the total wheat production?

Think step by step."""

result = chat_with_traces(cot_prompt)

print("=== Thinking Traces ===")
print(result["thinking"])
print("\n=== Final Answer ===")
print(result["answer"])

## 3. Mathematical Reasoning

Magistral models excel at multi-step mathematical problems. Let's test with a problem that requires careful algebraic reasoning.

In [ ]:
math_prompt = """A train leaves City A at 9:00 AM traveling at 80 km/h toward City B.
Another train leaves City B at 10:00 AM traveling at 100 km/h toward City A.
The cities are 500 km apart.

1. At what time do the trains meet?
2. How far from City A is the meeting point?
3. If a bird starts flying at 120 km/h from City A at 9:00 AM, bouncing back
   and forth between the two trains until they meet, what total distance does
   the bird fly?

Show your work for each question."""

result = chat_with_traces(math_prompt)

print("=== Thinking Traces ===")
print(result["thinking"][:2000])  # Truncated for readability
print("\n=== Final Answer ===")
print(result["answer"])

## 4. Logical Deduction

Let's test the model's ability to solve a classic logic puzzle that requires careful deductive reasoning.

In [ ]:
logic_prompt = """There are 4 people: Alice, Bob, Carol, and Dave.

Facts:
- One of them always tells the truth, one always lies, and the other two
  sometimes lie and sometimes tell the truth.
- Alice says: "Bob is the liar."
- Bob says: "Carol always tells the truth."
- Carol says: "Dave is not the truth-teller."
- Dave says: "Alice is not the liar."

We also know:
- The liar's statement is always false.
- The truth-teller's statement is always true.

Who is the truth-teller and who is the liar? Explain your reasoning step by step."""

result = chat_with_traces(logic_prompt)

print("=== Thinking Traces ===")
print(result["thinking"][:2000])
print("\n=== Final Answer ===")
print(result["answer"])

## 5. Code Analysis and Bug Detection

Reasoning models are particularly good at analyzing code, tracing execution paths, and identifying bugs. Let's give the model a piece of buggy code and ask it to find the issues.

In [ ]:
code_prompt = """Analyze the following Python function. It's supposed to find the
longest substring without repeating characters, but it has bugs. Identify all
bugs and provide the corrected version.

```python
def longest_unique_substring(s):
    max_len = 0
    start = 0
    seen = {}
    result = ""

    for end in range(len(s)):
        if s[end] in seen:
            start = seen[s[end]] + 1  # Bug: doesn't handle case where seen[s[end]] < start
        seen[s[end]] = end
        if end - start > max_len:  # Bug: should be >= for correct length calculation
            max_len = end - start
            result = s[start:end]  # Bug: should be s[start:end+1]

    return result

# Test cases:
# longest_unique_substring("abcabcbb") should return "abc"
# longest_unique_substring("bbbbb") should return "b"
# longest_unique_substring("pwwkew") should return "wke"
```

For each bug:
1. Explain what goes wrong
2. Provide a test case that triggers the bug
3. Show the fix"""

result = chat_with_traces(code_prompt)

print("=== Thinking Traces ===")
print(result["thinking"][:2000])
print("\n=== Final Answer ===")
print(result["answer"])

## 6. Comparison: Reasoning Model vs. Standard Model

Let's compare how a reasoning model (`magistral-medium-latest`) and a standard model (`mistral-large-latest`) handle the same complex problem. This highlights the value of extended reasoning for difficult tasks.

In [ ]:
comparison_prompt = """A snail is at the bottom of a 30-meter well. Each day, it climbs
up 3 meters, but each night, it slips back 2 meters. On what day does the snail
reach the top of the well?

Be careful — this is a classic trick question. Think through it carefully."""

# Reasoning model (Magistral)
print("=== Magistral (Reasoning Model) ===")
reasoning_result = chat_with_traces(comparison_prompt, model=REASONING_MODEL)
print("Thinking:")
print(reasoning_result["thinking"][:1500])
print("\nAnswer:")
print(reasoning_result["answer"])

print("\n" + "=" * 60 + "\n")

# Standard model
print("=== Mistral Large (Standard Model) ===")
standard_result = chat(comparison_prompt, model=STANDARD_MODEL)
print(standard_result)

## 7. Advanced: Controlling Reasoning with `prompt_mode`

Magistral models support a `prompt_mode` parameter to control their behavior:
- `"reasoning"` (default): Uses the built-in reasoning system prompt
- `None`: No system prompt — useful when providing your own instructions

You can also override the default system prompt by providing your own.

In [ ]:
custom_system = """You are a mathematics tutor. When solving problems:
1. State the given information
2. Identify what is being asked
3. Choose the appropriate method
4. Solve step by step
5. Verify your answer"""

problem = """Find all real solutions to the equation:
x^4 - 5x^2 + 4 = 0"""

result = chat_with_traces(problem, system=custom_system)

print("=== Thinking Traces ===")
print(result["thinking"][:1500])
print("\n=== Final Answer ===")
print(result["answer"])

## 8. Multi-Step Reasoning: Real-World Scenario

Let's test the model with a more realistic scenario that requires combining multiple types of reasoning.

In [ ]:
scenario_prompt = """A company is planning to launch a new product. Here are the constraints:

- The development team can work on at most 2 features per sprint (2 weeks).
- Feature A takes 1 sprint and has no dependencies.
- Feature B takes 2 sprints and depends on Feature A.
- Feature C takes 1 sprint and has no dependencies.
- Feature D takes 1 sprint and depends on both B and C.
- Feature E takes 2 sprints and depends on D.
- Testing takes 1 sprint after all features are complete.
- The team has a hard deadline of 12 weeks from now.

Questions:
1. What is the critical path?
2. What is the minimum number of sprints needed?
3. Can the team meet the 12-week deadline? If not, what would you recommend?
4. Draw a simple dependency graph using text/ASCII.

Think through this carefully, considering parallel execution where possible."""

result = chat_with_traces(scenario_prompt)

print("=== Thinking Traces ===")
print(result["thinking"][:2000])
print("\n=== Final Answer ===")
print(result["answer"])

## Best Practices for Reasoning with Mistral

### Model Selection
- Use `magistral-medium-latest` for complex problems requiring deep reasoning
- Use `magistral-small-latest` for simpler reasoning tasks with lower latency
- Use `mistral-large-latest` when reasoning is not the primary requirement

### Prompting Tips
- **Be explicit about reasoning**: Phrases like "Think step by step" or "Show your work" can improve output structure
- **Break down complex problems**: Provide clear structure with numbered sub-questions
- **Use custom system prompts** when you need the model to follow a specific reasoning framework
- **Leverage thinking traces**: Parse the `thinking` blocks to understand the model's reasoning process, useful for debugging and validation

### Understanding the Output
- Thinking traces (`type: "thinking"`) show the model's internal reasoning
- Final answers (`type: "text"`) contain the polished response
- The `prompt_mode` parameter controls whether the default reasoning system prompt is used

### When to Use Reasoning Models
- Mathematical problems requiring multiple steps
- Logic puzzles and constraint satisfaction
- Code analysis and debugging
- Complex planning and scheduling
- Any task where showing work improves accuracy

### Resources
- [Mistral Reasoning Documentation](https://docs.mistral.ai/capabilities/reasoning/)
- [Mistral API Reference](https://docs.mistral.ai/api/)
- [Mistral Cookbook](https://github.com/mistralai/cookbook)